# SELECCIÓ AUTOMÀTICA DE VEUS I ENTORNS

**Pas 4** del pipeline. Genera els .wav finals del dataset combinant tres eixos de diversitat:

1. **Veu** — locutors/es diferents, per clonatge (`ref_audio`).
2. **Entorn acústic** — sala, soroll ambiental i canal de transmissió.
3. **Prosòdia** — variació lleu de velocitat de parla.

La simulació acústica viu a **src/acoustic_sim.py** i segueix l'ordre físic real:

```
veu seca -> [SALA] -> [+ SOROLL] -> [CANAL] -> normalització
             RIR       SNR (dB)     filtre/còdec
```

Entrada: `lab/outputs/frases/dataset_entitats_whisper.jsonl`
(camps: `entity`, `raw_text`, `tts_text`, `style`).

Sortida: .wav a 24 kHz (màster) + còpia a 16 kHz (llesta per Whisper) i un manifest
amb totes les condicions aplicades a cada àudio.

## 0. Configuració, rutes i imports

In [ ]:
import json
import os
import random
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import soundfile as sf
import soxr
from IPython.display import Audio, display

# L'arrel es busca cap amunt des del directori de treball: escrita a mà, el projecte
# deixava d'arrencar en canviar de disc -- i aquesta encara apuntava al disc antic.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/llm.py").exists())
sys.path.insert(0, str(ROOT / "src"))

# Cache de HuggingFace dins del projecte: `~/.cache/huggingface` pertany a root en
# aquesta màquina i carregar OmniVoice hi peta amb PermissionError. Passar `cache_dir=`
# no serveix aquí: `OmniVoice.from_pretrained()` resol el model amb `snapshot_download()`
# sense `cache_dir`, així que només fa cas de la variable d'entorn. I ha d'estar posada
# ABANS d'importar `huggingface_hub`, que llegeix les rutes en importar-se: per això és
# en aquesta cel·la i no a la que carrega el model. Mateixa solució que
# `src/verify_entities.py`; `setdefault` respecta un HF_HOME propi.
os.environ.setdefault("HF_HOME", str(ROOT / ".hf_cache"))

import acoustic_sim as ac  # simulador acústic: sala + soroll + canal

# --- Entrades (sortida dels passos anteriors del pipeline) ---
DATASET_FRASES = ROOT / "lab/outputs/frases/dataset_entitats_whisper.jsonl"

# --- Sortides d'aquest pas ---
OUTPUT_DIR = ROOT / "datasets/audios/converses"           # màster 24 kHz
OUTPUT_DIR_16K = ROOT / "datasets/audios/converses_16k"   # còpia per a Whisper
MANIFEST_OUT = ROOT / "lab/outputs/converses/manifest_converses.jsonl"

# --- Recursos ---
AUDIOS_REFERENCIA_DIR = ROOT / "datasets/audios/referencia"  # .wav de veu clonada
# El banc de soroll i el de RIR els gestiona acoustic_sim (ac.DIR_SOROLL / ac.DIR_RIR).
# Si el banc de soroll no hi és, executa:  python3 src/build_noise_bank.py

SEED = 42
random.seed(SEED)


def triar_dispositiu():
    """Tria la primera GPU que aquesta build de PyTorch pugui fer servir de debò.

    Aquesta màquina té dues GPU i `cuda:0` a cegues no serveix: la GTX 1080 Ti és
    Pascal (sm_61) i les builds de torch amb CUDA recent ja no la compilen, així que
    peta amb "no kernel image is available for execution on the device" només quan
    ja s'han carregat els pesos. En comptes de comparar capacitats a mà, provem una
    operació petita a cada dispositiu i ens quedem amb el primer que respon.
    L'única que serveix aquí és la RTX 4060 Ti. `src/verify_entities.py` fa el mateix
    des de `triar_dispositiu()`.
    """
    if not torch.cuda.is_available():
        return "cpu"
    for i in range(torch.cuda.device_count()):
        try:
            torch.zeros(8, device=f"cuda:{i}").sum().item()
            return f"cuda:{i}"
        except Exception as exc:
            nom = torch.cuda.get_device_name(i)
            print(f"  cuda:{i} ({nom}) descartada: {type(exc).__name__}")
    return "cpu"


DEVICE = triar_dispositiu()
SAMPLE_RATE = 24000            # màster: OmniVoice genera nativament a 24 kHz
SAMPLE_RATE_WHISPER = 16000    # el que consumeix Whisper
IDIOMA = "es"

print(f"Dispositiu: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(DEVICE)})" if DEVICE.startswith("cuda") else ""))
print(f"Banc de soroll: {'OK' if ac.DIR_SOROLL.exists() else 'NO TROBAT (fallback sintètic)'}")

Dispositiu: cuda:0 (NVIDIA GeForce RTX 4060 Ti)
Banc de soroll: OK


/home/ugiat/.virtualenvs/sintetic/lib/python3.10/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU1 NVIDIA GeForce GTX 1080 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/ugiat/.virtualenvs/sintetic/lib/python3.10/site-packages/torch/cuda/__init__.py:502: UserWarning: 
NVIDIA GeForce GTX 1080 Ti with CUDA capability sm_61 is not compatible with the current PyTorch

## 1. Càrrega del dataset de frases

Reutilitzem la sortida de `generate_sentences.ipynb`. Cada línia és una frase per a una
entitat concreta, amb la seva versió ortogràfica (`raw_text`) i la versió fonètica per a
TTS (`tts_text`).


In [2]:
def carregar_dataset_frases(path):
    """Carrega el .jsonl de frases (entity, raw_text, tts_text, style)."""
    frases = []
    with open(path, "r", encoding="utf-8") as f:
        for linia in f:
            linia = linia.strip()
            if linia:
                frases.append(json.loads(linia))
    return frases


frases = carregar_dataset_frases(DATASET_FRASES)
print(f"Frases carregades: {len(frases)}")
frases[:3]


Frases carregades: 26


[{'entity': 'Hollywood',
  'raw_text': 'Los premios de Hollywood se celebrarán este fin de semana.',
  'tts_text': 'Los premios de Jólivuud se celebrarán este fin de semana.',
  'style': 'Titular de última hora (frases cortas y directas)'},
 {'entity': 'Hollywood',
  'raw_text': 'Las películas de Hollywood han batido récords de taquilla en todo el mundo.',
  'tts_text': 'Las películas de Jólivuud han batido récords de taquilla en todo el mundo.',
  'style': 'Entradilla del presentador en plató (tono formal e introductorio)'},
 {'entity': 'Hollywood',
  'raw_text': 'Las premiaciones de Hollywood atraen a millones de espectadores cada año.',
  'tts_text': 'Las premiaciones de Jólivuud atraen a millones de espectadores cada año.',
  'style': 'Declaraciones en una rueda de prensa o debate (estilo más hablado)'}]

## 2. Pool de veus (OmniVoice)

OmniVoice admet una segona estratègia (`instruct`, descripció textual del locutor)
a banda de `ref_audio`, però l'hem descartada: en castellà, l'`instruct` sempre surt
amb accent llatinoamericà encara que no se li demani, i no hi ha manera de forçar
accent peninsular des del vocabulari tancat que accepta. Fem servir només **clonatge
a partir d'àudio de referència** (`ref_audio`).

Dues fonts de referència, amb un tractament diferent de l'entorn acústic:

1. **Ja ambientades** (`v1..v9.wav`, de VoxPopuli): àudio real, no net, amb la seva
   pròpia sala i soroll de fons. En clonar, el timbre arrossega aquest ambient (els
   encoders de veu no el separen del tot), així que aplicar-hi el simulador acústic
   a sobre l'acumularia (reverberació sobre reverberació, soroll sobre soroll) i, de
   retruc, trencaria l'aparellament lliure amb qualsevol entorn: cada veu quedaria
   lligada a l'ambient amb què es va gravar. Es marquen `font_real=True`, i
   `construir_manifest` (secció 4) les fixa a l'entorn `font_real` (sense sala,
   soroll ni canal, només normalització de nivell) en comptes de fer-hi round-robin
   amb la resta.
2. **Netes** (`common_voice_*.wav`, de Mozilla Common Voice, accent peninsular
   filtrat via `src/build_voice_bank.py`): gravades sense soroll ni sala. Passen pel
   simulador acústic sencer (secció 3): la mateixa veu es pot combinar amb qualsevol
   entorn, cosa que manté ortogonals els dos eixos de diversitat (veu i entorn) i
   deixa el manifest auditable (SNR, RIR i canal exactes de cada àudio). Es carreguen
   dinàmicament de `datasets/audios/referencia/locutors.json` (escrit pel mateix
   script) perquè ampliar-les no obligui a tocar aquest notebook.

Definim un registre (`VEUS`) amb totes les veus disponibles perquè la resta del
notebook les tracti de manera uniforme. De moment el dataset de frases
(`dataset_entitats_whisper_1.jsonl`) és tot en castellà, així que només hi ha veus
`idioma="es"`.


In [ ]:
# Cada veu és un dict amb:
#   id         -> identificador curt, servirà per anomenar fitxers
#   tipus      -> "ref_audio" (únic tipus admès, veure secció 2)
#   valor      -> ruta al .wav de referència
#   idioma     -> "es" | "ca" (per triar l'àudio de referència / entitats correctes)
#   font_real  -> True si la referència ja porta condicions acústiques reals
#                 barrejades (VoxPopuli): `construir_manifest` la fixa a l'entorn
#                 `font_real` en comptes de fer-hi round-robin amb la resta.
#   metadata   -> info lliure (gènere, edat, accent...) útil per filtrar/balancejar

VEUS = [
    # --- Referències ja ambientades (VoxPopuli): fixades a l'entorn "font_real" ---
    *[
        {"id": f"ref_voxpopuli_{n:02d}", "tipus": "ref_audio",
         "valor": str(AUDIOS_REFERENCIA_DIR / f"v{n}.wav"),
         "idioma": "es", "font_real": True, "metadata": {"font": "voxpopuli"}}
        for n in range(1, 10)
    ],
]

# --- Referències netes (Common Voice, veure src/build_voice_bank.py): passen pel
# simulador acústic sencer. Es carreguen de locutors.json (generat pel mateix
# script) perquè ampliar el banc de veus no obligui a tocar aquest notebook.
_locutors_cv = AUDIOS_REFERENCIA_DIR / "locutors.json"
if _locutors_cv.exists():
    with open(_locutors_cv, encoding="utf-8") as f:
        _metadata_cv = json.load(f)
    VEUS += [
        {"id": f"ref_{nom}", "tipus": "ref_audio",
         "valor": str(AUDIOS_REFERENCIA_DIR / f"{nom}.wav"),
         "idioma": "es", "font_real": False,
         "metadata": {"font": "common_voice", **meta}}
        for nom, meta in _metadata_cv.items()
    ]


# La selecció de veus la fa `construir_manifest` (secció 4) amb round-robin, que
# reparteix les veus de manera equitativa. Un `random.choice` per frase deixava
# desequilibris grossos per pura sort amb pocs elements.

print(f"Veus disponibles: {len(VEUS)} "
      f"({sum(v['font_real'] for v in VEUS)} font_real, "
      f"{sum(not v['font_real'] for v in VEUS)} netes)")

## 3. Matriu d'entorns acústics

Cada entorn combina fins a tres capes, en l'ordre en què passen a la realitat:

| Capa | Què fa | Com |
|---|---|---|
| **Sala** | la veu rebota a l'espai | convolució amb una RIR generada per `pyroomacoustics` (mètode de les fonts imatge) |
| **Soroll** | ambient del lloc | mescla additiva a una **SNR controlada en dB**, mesurant la veu només sobre les trames actives (fer-ho sobre el senyal sencer infla el soroll perquè els silencis baixen l'RMS) |
| **Canal** | micròfon + transmissió | cadena de `pedalboard`: filtres, distorsió i **còdecs reals** (GSM 06.10 per al telèfon, MP3 per al mòbil) |

El soroll ve de **DEMAND** (gravacions reals d'ambients, CC BY 4.0) i les RIRs es generen
paramètricament, així que no depenem de cap banc de reverberacions descarregat.

Si el banc de soroll no està descarregat (`python3 src/build_noise_bank.py`),
`acoustic_sim` recorre a soroll procedural o a murmuri construït superposant els
propis àudios TTS, de manera que el pipeline no es bloqueja mai.

In [4]:
from acoustic_sim import ENTORNS, SALES, entorns_per_style

# El banc de RIR es genera un sol cop i es desa a disc (~2 s, 36 fitxers).
# Es guarda perquè el dataset sigui auditable: si algú vol saber com sonava la
# sala d'un àudio concret, pot escoltar directament la RIR que indica el manifest.
ac.construir_banc_rir()

print(f"\n{len(ENTORNS)} entorns definits:\n")
capcalera = f"{'id':16s} {'pes':>4s} {'sala':>12s} {'RT60':>6s} {'ambient':>12s} {'SNR dB':>8s} {'canal':>14s}"
print(capcalera)
print("-" * len(capcalera))
for e in ENTORNS.values():
    snr = f"{e.snr_db[0]:.0f}-{e.snr_db[1]:.0f}" if e.snr_db else "-"
    rt60 = f"{SALES[e.sala].rt60_real:.2f}s" if e.sala else "-"
    print(f"{e.id:16s} {e.pes:4.1f} {str(e.sala or '-'):>12s} {rt60:>6s} "
          f"{str(e.ambient or '-'):>12s} {snr:>8s} {str(e.canal or '-'):>14s}")

print("\nAfinitat entorn <-> style (un entorn incoherent amb l'estil resta realisme):")
for style in sorted({f.get("style", "") for f in frases}):
    noms = ", ".join(e.id for e in entorns_per_style(style))
    print(f"  {style[:52]:52s} -> {noms}")

  plato_tv       6 variants  (RT60 ~0.29s, RIR de 0.46s)
  despatx        6 variants  (RT60 ~0.28s, RIR de 0.22s)
  redaccio       6 variants  (RT60 ~0.46s, RIR de 0.62s)
  sala_premsa    6 variants  (RT60 ~0.70s, RIR de 0.68s)
  sala_actes     6 variants  (RT60 ~1.31s, RIR de 1.29s)
  passadis       6 variants  (RT60 ~0.94s, RIR de 0.87s)

10 entorns definits:

id                pes         sala   RT60      ambient   SNR dB          canal
------------------------------------------------------------------------------
estudi_net        2.0            -      -            -        -         estudi
plato_tv          2.0     plato_tv  0.29s     redaccio    28-38 micro_faristol
redaccio          1.5     redaccio  0.46s     redaccio    14-24  micro_corbata
sala_premsa       1.5  sala_premsa  0.70s  sala_premsa    16-26 micro_faristol
sala_actes        0.8   sala_actes  1.31s  sala_premsa    20-30 micro_faristol
carrer            1.5            -      -       carrer     8-18  micro_corbata
ext

## 4. Manifest: assignació equilibrada de veu i entorn

Combinem frases, veus i entorns en un manifest reproduïble (`SEED`) que descriu
exactament què cal generar abans de tocar la GPU.

L'assignació **no és aleatòria pura**: amb poques frases i poques veus, l'atzar deixa
desequilibris grossos per pura sort (una veu que surt el triple que una altra). Fem
round-robin sobre la llista sencera, remenant-la a cada volta.

In [ ]:
def construir_manifest(frases, veus, entorns=ENTORNS, idioma=IDIOMA, seed=SEED):
    """Assigna veu, entorn i velocitat a cada frase de manera equilibrada.

    Quatre criteris:
      1. Les veus es reparteixen per round-robin: cap veu no pot sortir més d'un
         cop més que qualsevol altra.
      2. L'entorn ha de ser coherent amb l'`style` de la frase (una crònica de
         corresponsal sona a carrer o a telèfon, no a plató).
      3. Dins dels entorns compatibles amb l'estil, es respecten els pesos.
      4. Una veu `font_real` (ja ambientada, veure secció 2) es fixa sempre a
         l'entorn `font_real`: no té sentit fer-hi round-robin amb la resta
         perquè no pot combinar-se amb cap altre entorn sense duplicar-lo.
         Per això queda fora del pool general de la resta de veus (punts 2 i 3):
         `expandir_per_pes` posa un mínim d'una còpia encara que el pes sigui 0,
         així que sense excloure'l explícitament es colaria per a qualsevol frase
         l'`style` de la qual no encaixés amb cap altre entorn.
    """
    rng = np.random.default_rng(seed)

    candidates = [v for v in veus if v["idioma"] == idioma]
    if not candidates:
        raise ValueError(f"Cap veu disponible per idioma={idioma!r}")

    # 1. Veus: una passada round-robin per a tot el dataset.
    veus_assignades = ac.mostreig_round_robin(candidates, len(frases), rng)

    # 2 i 3. Entorns: round-robin independent dins de cada estil, sobre la llista
    # d'entorns compatibles ja expandida segons el pes de cadascun. "font_real"
    # només s'assigna per l'override del punt 4, mai per aquest round-robin.
    entorns_normals = {k: v for k, v in entorns.items() if k != "font_real"}
    per_style = {}
    for idx, frase in enumerate(frases):
        per_style.setdefault(frase.get("style", ""), []).append(idx)

    entorn_de = {}
    for style, indexs in per_style.items():
        compatibles = ac.expandir_per_pes(ac.entorns_per_style(style, entorns_normals))
        for idx, entorn in zip(indexs, ac.mostreig_round_robin(compatibles, len(indexs), rng)):
            entorn_de[idx] = entorn

    manifest = []
    for i, (frase, veu) in enumerate(zip(frases, veus_assignades)):
        # 4. Override: una veu font_real salta el round-robin d'entorns.
        entorn = entorns["font_real"] if veu.get("font_real") else entorn_de[i]
        manifest.append({
            **frase,
            "conversa_id": f"conv_{i:05d}",
            "veu_id": veu["id"],
            "entorn_id": entorn.id,
            # Velocitat lleugerament variable: diversitat prosòdica sense que la
            # frase deixi de sonar natural.
            "speed": round(float(rng.uniform(0.92, 1.08)), 3),
        })
    return manifest


manifest = construir_manifest(frases, VEUS)
print(f"Manifest generat: {len(manifest)} entrades\n")

def repartiment(titol, comptador, total):
    print(f"{titol} ({len(comptador)} valors, ideal {total/len(comptador):.1f} c/u)")
    for clau, n in comptador.most_common():
        print(f"   {clau:18s} {n:3d}  {'#' * n}")

repartiment("Per veu:", Counter(m["veu_id"] for m in manifest), len(manifest))
print()
repartiment("Per entorn:", Counter(m["entorn_id"] for m in manifest), len(manifest))

## 5. Càrrega del model OmniVoice

In [6]:
def carregar_model_omnivoice(device=DEVICE):
    from omnivoice import OmniVoice

    dtype = torch.float16 if device.startswith("cuda") else torch.float32
    model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
    return model


model = carregar_model_omnivoice()


/home/ugiat/.virtualenvs/sintetic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 95659.56it/s]


Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]


Loading weights:  34%|███▍      | 106/313 [00:00<00:00, 1041.09it/s]

...


Loading weights: 100%|██████████| 313/313 [00:00<00:00, 1045.88it/s]


Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]


Loading weights:  46%|████▌     | 240/527 [00:00<00:00, 2390.71it/s]


Loading weights:  98%|█████████▊| 514/527 [00:00<00:00, 2535.68it/s]


Loading weights: 100%|██████████| 527/527 [00:00<00:00, 2527.36it/s]

## 6. Generació TTS per veu

Encapsula la crida a `model.generate(...)` amb `ref_audio` (patró vist a
`omnivoice.ipynb` i `src/generate_omnivoice_tts.py`).


In [ ]:
def generar_tts(model, text, veu, speed=None, idioma=IDIOMA):
    """Genera l'àudio (numpy array) d'una frase amb la veu indicada.

    Passem `language` explícitament perquè la documentació d'OmniVoice indica que
    el rendiment és una mica millor que en mode agnòstic d'idioma.
    """
    if veu["tipus"] != "ref_audio":
        raise ValueError(f"Tipus de veu desconegut: {veu['tipus']}")

    kwargs = {"text": text, "language": idioma, "ref_audio": veu["valor"]}
    if speed is not None:
        kwargs["speed"] = float(speed)

    return model.generate(**kwargs)[0]

## 7. Mescla amb l'entorn

Tota la feina la fa `src/acoustic_sim.py`. Aquí només derivem una llavor estable per
frase, perquè cada àudio rebi sempre les mateixes condicions encara que es reprengui
una generació a mitges o es reordeni el dataset.

In [8]:
def aplicar_entorn(waveform, entrada):
    """Aplica sala + soroll + canal a l'àudio d'una entrada del manifest.

    Retorna (àudio, metadades). Les metadades porten els valors concrets que
    s'han fet servir (SNR exacta, quina RIR, quin canal) perquè el manifest els
    registri i el dataset sigui auditable.
    """
    # Llavor derivada del contingut i no de l'ordre d'iteració: així un `--resume`
    # o un canvi d'ordre de les frases no altera el que ja s'havia generat.
    rng = np.random.default_rng(
        ac.llavor_estable(SEED, entrada["conversa_id"], entrada["entorn_id"]) % 2**31
    )
    return ac.mesclar_amb_entorn(
        waveform, SAMPLE_RATE, ENTORNS[entrada["entorn_id"]], rng,
        dir_veus_babble=OUTPUT_DIR,  # murmuri en castellà si falta el banc DEMAND
    )

## 8. Pipeline complet

Itera el manifest, genera l'àudio, aplica l'entorn, desa el màster a 24 kHz i la còpia a
16 kHz per a Whisper, i registra a cada entrada les condicions acústiques aplicades.

In [9]:
def desar_audio(x, ruta, sr):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    sf.write(ruta, x, sr, subtype="PCM_16")


def carregar_manifest_previ(path=MANIFEST_OUT):
    """Metadades de l'execució anterior, indexades per `conversa_id`."""
    path = Path(path)
    if not path.exists():
        return {}
    previ = {}
    with open(path, encoding="utf-8") as f:
        for linia in f:
            if linia.strip():
                entrada = json.loads(linia)
                previ[entrada["conversa_id"]] = entrada
    return previ


def generar_dataset_converses(model, manifest, output_dir=OUTPUT_DIR,
                              output_dir_16k=OUTPUT_DIR_16K, overwrite=False,
                              limit=None):
    veus_per_id = {v["id"]: v for v in VEUS}
    entrades = manifest[:limit] if limit else manifest

    # En reprendre una generació a mitges, els .wav que ja hi són se salten. Cal
    # recuperar-ne les metadades del manifest anterior: si no, el manifest nou
    # sortiria sense `audio_path` ni `snr_db` per a tot el que ja estava fet i
    # perdríem el registre de les condicions acústiques d'aquells àudios.
    previ = carregar_manifest_previ() if not overwrite else {}

    resultats = []
    for i, entrada in enumerate(entrades, start=1):
        entrada = dict(entrada)  # no mutem el manifest d'entrada
        out_path = Path(output_dir) / f"{entrada['conversa_id']}.wav"

        if out_path.exists() and not overwrite:
            anterior = previ.get(entrada["conversa_id"], {})
            if anterior.get("status") == "ok":
                entrada.update({k: v for k, v in anterior.items() if k not in entrada
                                or entrada[k] is None})
                entrada["status"] = "ok"
                print(f"[{i}/{len(entrades)}] skip {entrada['conversa_id']} (metadades recuperades)")
            else:
                # Hi ha el .wav però no en sabem les condicions: ho fem constar en
                # comptes de deixar una entrada muda al manifest.
                info = sf.info(out_path)
                entrada["audio_path"] = str(out_path.relative_to(ROOT))
                entrada["durada_s"] = round(info.duration, 3)
                entrada["sample_rate"] = info.samplerate
                entrada["status"] = "skip_sense_metadades"
                print(f"[{i}/{len(entrades)}] skip {entrada['conversa_id']} "
                      f"(ATENCIÓ: sense metadades acústiques; cal --overwrite per regenerar)")
            resultats.append(entrada)
            continue

        try:
            veu = veus_per_id[entrada["veu_id"]]
            waveform = generar_tts(model, entrada["tts_text"], veu, speed=entrada.get("speed"))
            audio, meta = aplicar_entorn(waveform, entrada)

            if audio.size == 0:
                raise ValueError("OmniVoice ha retornat un àudio buit")

            desar_audio(audio, out_path, SAMPLE_RATE)
            audio_16k = soxr.resample(audio, SAMPLE_RATE, SAMPLE_RATE_WHISPER, quality="VHQ")
            desar_audio(audio_16k, Path(output_dir_16k) / f"{entrada['conversa_id']}.wav",
                        SAMPLE_RATE_WHISPER)

            entrada.update(meta)  # sala, rir_path, ambient, snr_db, canal, nivell_dbfs
            entrada["audio_path"] = str(out_path.relative_to(ROOT))
            entrada["audio_path_16k"] = str((Path(output_dir_16k) /
                                             f"{entrada['conversa_id']}.wav").relative_to(ROOT))
            entrada["durada_s"] = round(len(audio) / SAMPLE_RATE, 3)
            entrada["sample_rate"] = SAMPLE_RATE
            entrada["status"] = "ok"

            snr = f"SNR {meta['snr_db']:>4.1f} dB" if meta["snr_db"] is not None else "sense soroll"
            print(f"[{i}/{len(entrades)}] ok {entrada['conversa_id']}  "
                  f"{entrada['veu_id'][:26]:26s} {entrada['entorn_id']:15s} {snr}  "
                  f"{entrada['durada_s']:.1f}s")

        except Exception as exc:
            entrada["status"] = "error"
            entrada["error"] = f"{type(exc).__name__}: {exc}"
            print(f"[{i}/{len(entrades)}] ERROR {entrada['conversa_id']}: {exc}")

        resultats.append(entrada)

    return resultats


def desar_manifest(manifest, path=MANIFEST_OUT):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for entrada in manifest:
            f.write(json.dumps(entrada, ensure_ascii=False) + "\n")
    estats = Counter(e.get("status") for e in manifest)
    print(f"\nManifest desat a: {path}  ({dict(estats)})")

In [10]:
# Dataset complet. Per a una prova pilot ràpida abans de llançar-ho tot, passa
# `limit=6`: genera només les primeres frases, que ja cobreixen diversos entorns.
resultats = generar_dataset_converses(model, manifest, overwrite=True)
desar_manifest(resultats)

[1/26] ok conv_00000  instruct_male_young_low    redaccio        SNR 21.1 dB  3.3s


[2/26] ok conv_00001  instruct_female_senior_neu plato_tv        SNR 32.7 dB  4.7s



Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 131387.84it/s]


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

[22/26] ok conv_00021  instruct_female_young_high estudi_net      sense soroll  2.9s


[23/26] ok conv_00022  instruct_male_senior_low   plato_tv        SNR 33.1 dB  4.2s


[24/26] ok conv_00023  instruct_male_middle_neutr cafeteria       SNR  7.4 dB  4.4s


[25/26] ok conv_00024  instruct_female_senior_neu connexio_movil  SNR 17.7 dB  5.1s


[26/26] ok conv_00025  ref_locutor_es_01          connexio_movil  SNR 14.2 dB  3.8s

Manifest desat a: /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/outputs/converses/manifest_converses.jsonl  ({'ok': 26})


## 9. Verificació

Dues comprovacions abans de donar el dataset per bo: que els senyals són sans
(sense NaN, sense saturació, amb rang dinàmic raonable) i que sonen com toca.

In [11]:
def verificar(manifest):
    """Comprova la salut dels .wav generats: NaN, clipping i rang dinàmic."""
    files = []
    for entrada in manifest:
        if entrada.get("status") != "ok":
            continue
        x, sr = sf.read(ROOT / entrada["audio_path"], dtype="float32")
        pic = float(np.abs(x).max())
        files.append({
            "id": entrada["conversa_id"], "entorn": entrada["entorn_id"],
            "pic": pic, "rms": ac.rms(x), "crest_db": 20 * np.log10(pic / ac.rms(x)),
            "clip": int((np.abs(x) >= 0.9999).sum()),
            "nan": bool(np.isnan(x).any() or np.isinf(x).any()),
            "snr_db": entrada.get("snr_db"), "durada": entrada.get("durada_s"),
        })

    print(f"{'id':12s} {'entorn':16s} {'dur':>5s} {'pic':>6s} {'crest':>7s} {'SNR':>6s} "
          f"{'clip':>5s} {'NaN':>4s}")
    for f in files:
        snr = f"{f['snr_db']:.1f}" if f["snr_db"] is not None else "-"
        print(f"{f['id']:12s} {f['entorn']:16s} {f['durada']:5.1f} {f['pic']:6.3f} "
              f"{f['crest_db']:6.1f}dB {snr:>6s} {f['clip']:5d} {'SI' if f['nan'] else 'no':>4s}")

    problemes = [f for f in files if f["nan"] or f["clip"] > 0 or f["pic"] > 1.0]
    print(f"\n{'Tot correcte' if not problemes else 'PROBLEMES: ' + str([f['id'] for f in problemes])}")
    return files


_ = verificar(resultats)

id           entorn             dur    pic   crest    SNR  clip  NaN
conv_00000   redaccio           3.3  0.467   12.6dB   21.1     0   no
conv_00001   plato_tv           4.7  0.261   14.8dB   32.7     0   no
conv_00002   sala_premsa        3.4  0.648   17.3dB   21.2     0   no
conv_00003   carrer             7.1  0.252   13.0dB    9.6     0   no
conv_00004   estudi_net         3.8  0.330   14.2dB      -     0   no
conv_00005   connexio_movil     4.1  0.394   12.0dB   10.3     0   no
conv_00006   redaccio           4.1  0.342   12.6dB   19.0     0   no
conv_00007   telefon            7.9  0.625   19.5dB   12.8     0   no
conv_00008   plato_tv           4.2  0.439   16.9dB   28.2     0   no
conv_00009   sala_actes         4.3  0.607   15.9dB   23.0     0   no
conv_00010   connexio_movil     5.2  0.435   11.8dB   12.7     0   no
conv_00011   estudi_net         3.2  0.295   12.0dB      -     0   no
conv_00012   plato_tv           3.9  0.474   14.8dB   36.0     0   no
conv_00013   redaccio

In [12]:
def escoltar_mostres(manifest, n=6):
    """Reprodueix una mostra de cada entorn generat."""
    vistos, mostres = set(), []
    for entrada in manifest:
        if entrada.get("status") == "ok" and entrada["entorn_id"] not in vistos:
            vistos.add(entrada["entorn_id"])
            mostres.append(entrada)
    for entrada in mostres[:n]:
        snr = f", SNR {entrada['snr_db']} dB" if entrada.get("snr_db") is not None else ""
        sala = f", sala {entrada['sala']}" if entrada.get("sala") else ""
        print(f"[{entrada['conversa_id']}] {entrada['entorn_id']}"
              f" (veu {entrada['veu_id']}{sala}{snr})")
        print(f"  {entrada['raw_text']}")
        display(Audio(filename=str(ROOT / entrada["audio_path"])))


escoltar_mostres(resultats)

[conv_00000] redaccio (veu instruct_male_young_low, sala redaccio, SNR 21.14 dB)
  Los premios de Hollywood se celebrarán este fin de semana.


[conv_00001] plato_tv (veu instruct_female_senior_neutral, sala plato_tv, SNR 32.74 dB)
  Las películas de Hollywood han batido récords de taquilla en todo el mundo.


[conv_00002] sala_premsa (veu ref_locutor_es_01, sala sala_premsa, SNR 21.17 dB)
  Las premiaciones de Hollywood atraen a millones de espectadores cada año.


[conv_00003] carrer (veu instruct_male_middle_neutral, SNR 9.63 dB)
  Las producciones de Hollywood han marcado un antes y un después en la historia del cine, influyendo en diversas culturas.


[conv_00004] estudi_net (veu instruct_female_middle_neutral)
  La startup recibió un premio por su contribución a la sostenibilidad.


[conv_00005] connexio_movil (veu instruct_male_senior_low, SNR 10.27 dB)
  Hollywood ha anunciado la fecha de estreno de su nueva película.
